In [1]:
# --- Reproducibility Seed Setup ---
import os
import random
import numpy as np
import torch

SEED = 3180

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Global random seed set:", SEED)

Global random seed set: 3180


In [2]:
import sys, site, platform
print("PY:", platform.python_version())
print("EXE:", sys.executable)
print("USER_SITE:", site.getusersitepackages())


PY: 3.9.18
EXE: /common/software/install/manual/pytorch/2.1.2-pyclustertend/bin/python
USER_SITE: /users/4/volko028/.local/lib/python3.9/site-packages


In [3]:
# !python -m pip install torch transformers accelerate

In [4]:
# # Use the notebook's interpreter, not the system default
# !"{sys.executable}" -m pip install -U --no-cache-dir transformers==4.45.2 accelerate>=0.33

# # If you also need to ensure your CUDA torch is in THIS interpreter:
# !"{sys.executable}" -m pip install -U --no-cache-dir \
#   --index-url https://download.pytorch.org/whl/cu128 \
#   torch==2.9.0


In [5]:
import torch, transformers, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.1.2
transformers: 4.45.2
accelerate: 1.10.1


In [6]:
import os, torch, platform
print("python:", platform.python_version())
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("cuda.is_available():", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count(), "visible:", os.environ.get("CUDA_VISIBLE_DEVICES"))

python: 3.9.18
torch: 2.1.2 cuda: 11.8
cuda.is_available(): True
device_count: 1 visible: 0


In [7]:
# --- Imports ---
from sklearn.model_selection import train_test_split
import math, torch, pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score
)
from scipy.special import expit
import numpy as np

2026-04-06 17:22:12.270174: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-06 17:22:12.270232: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-06 17:22:12.270240: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-06 17:22:12.276809: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
# --- Config ---
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN    = 256
PER_DEVICE_BS = 2
GRAD_ACCUM    = 4
NUM_EPOCHS    = 3
EVAL_EVERY    = 2000
SAVE_EVERY    = 2000
STRIDE=32

In [9]:
# --- Load data ---
import pandas as pd

use_cols = ["notes", "outcome_hospitalization"]
train_df = pd.read_csv("mv_train_DISPOSITION.csv", usecols = use_cols)
val_df   = pd.read_csv("mv_val_DISPOSITION.csv", usecols = use_cols)
test_df  = pd.read_csv("mv_test_DISPOSITION.csv", usecols = use_cols)

In [10]:
# --- Tokenizer & collator ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

from transformers import DataCollatorWithPadding
import torch

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def tokenize_with_stride(text: str):
    text = text[:6000]
    return tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        padding=False
    )


In [11]:
# --- Streaming Dataset Loader ---
import math
import pandas as pd
import torch
import random
import numpy as np

class StreamingNotes(torch.utils.data.IterableDataset):
    def __init__(
        self,
        csv_path: str,
        dup_pos: int = 1,
        shuffle_within_chunk: bool = True,
        seed: int = 0,
        tokenizer=None,
        length_cache=None,
        max_len: int = 256,
        stride: int = 32,
        max_char: int = 6000,
        chunksize: int = 8192,
        text_col: str = "notes",
        label_col: str = "outcome_hospitalization",
    ):
        super().__init__()
        self.csv_path = csv_path
        self.dup_pos = max(1, int(dup_pos))
        self.shuffle_within_chunk = shuffle_within_chunk
        self.seed = seed

        self.tokenizer = tokenizer
        self.max_len = int(max_len)
        self.stride = int(stride)
        self.max_char = int(max_char)
        self.chunksize = int(chunksize)

        self.text_col = text_col
        self.label_col = label_col

        self._length = length_cache
        if self._length is None and self.tokenizer is not None:
            self._length = self._estimate_length()

    def __len__(self):
        if self._length is None:
            raise TypeError(
                "StreamingNotes has no length. Pass tokenizer=... so it can estimate __len__, "
                "or pass length_cache=<int>."
            )
        return int(self._length)

    def _tokenize_batch(self, texts):
        return self.tokenizer(
            texts,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            stride=self.stride,
            return_overflowing_tokens=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            padding=False
        )

    def _estimate_length(self):
        total = 0
        usecols = [self.text_col, self.label_col]

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            texts = chunk[self.text_col].astype(str).str.slice(0, self.max_char).tolist()
            labels = chunk[self.label_col].astype(int).to_numpy()

            enc = self._tokenize_batch(texts)
            mapping = enc.get("overflow_to_sample_mapping", None)

            if mapping is None:
                for y in labels:
                    repeats = self.dup_pos if y == 1 else 1
                    total += repeats
            else:
                mapping = np.asarray(mapping)
                for j in range(len(texts)):
                    n_win = int(np.sum(mapping == j))
                    repeats = self.dup_pos if labels[j] == 1 else 1
                    total += repeats * n_win

        return int(total)

    def __iter__(self):
        rng = random.Random(self.seed)
        usecols = [self.text_col, self.label_col]

        global_note_id = 0

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            rows = list(chunk.iterrows())
            if self.shuffle_within_chunk:
                rng.shuffle(rows)

            for _, row in rows:
                y = int(row[self.label_col])
                text = str(row[self.text_col])[:self.max_char]

                enc = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=True,
                    max_length=self.max_len,
                    stride=self.stride,
                    return_overflowing_tokens=True,
                    return_attention_mask=True,
                    return_token_type_ids=True,
                    padding=False
                )

                n = len(enc["input_ids"])
                repeats = self.dup_pos if y == 1 else 1

                for _ in range(repeats):
                    for k in range(n):
                        item = {
                            "input_ids": enc["input_ids"][k],
                            "attention_mask": enc["attention_mask"][k],
                            "labels": y,
                            "note_id": global_note_id,
                        }
                        if "token_type_ids" in enc:
                            item["token_type_ids"] = enc["token_type_ids"][k]
                        yield item

                global_note_id += 1
                
                
                
train_ds = StreamingNotes(
    "mv_train_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=True,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

val_ds = StreamingNotes(
    "mv_val_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)



In [12]:
# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: "NoAdmit", 1: "Admit"}
model.config.label2id = {"NoAdmit": 0, "Admit": 1}

/common/software/install/manual/pytorch/2.1.2-pyclustertend/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# --- Training Arguments ---
n_train = len(train_ds)
world_size = max(1, torch.cuda.device_count())
effective_bsz = PER_DEVICE_BS * GRAD_ACCUM * world_size
steps_per_epoch = math.ceil(n_train / effective_bsz)
MAX_STEPS = steps_per_epoch * NUM_EPOCHS
print(f"Training rows={n_train}  steps/epoch={steps_per_epoch}  max_steps={MAX_STEPS}")
assert MAX_STEPS > 0, "No training rows found. Check CSV path and columns."

training_args = TrainingArguments(
    output_dir="runs/clin-longformer",
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=-1,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_accumulation_steps=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=SAVE_EVERY,
    load_best_model_at_end=True,
    metric_for_best_model="note_auroc",
    greater_is_better=True,
    save_total_limit=3,
    remove_unused_columns=False,
    include_inputs_for_metrics=True,
    report_to="none",
    gradient_checkpointing=True,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Training rows=152685  steps/epoch=19086  max_steps=57258


In [14]:
# --- Create Trainer ---
import numpy as np
import torch
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score

class NoteAwareTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        inputs.pop("note_id", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)


    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        note_ids = inputs.pop("note_id", None)

        loss, logits, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys
        )

        if note_ids is not None:
            if not isinstance(note_ids, torch.Tensor):
                note_ids = torch.as_tensor(note_ids, dtype=torch.long)
            note_ids = note_ids.view(-1, 1)
            return loss, (logits, note_ids), labels

        return loss, logits, labels

POOL = "mean"

def compute_metrics_note_level(eval_pred):
    preds_obj, labels = eval_pred.predictions, eval_pred.label_ids
    logits, note_ids = preds_obj
    if isinstance(logits, torch.Tensor): logits = logits.detach().cpu().numpy()
    if isinstance(note_ids, torch.Tensor): note_ids = note_ids.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor): labels = labels.detach().cpu().numpy()
    note_ids = note_ids.squeeze(-1)

    logits = np.asarray(logits)
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)
    p1 = probs[:, 1]

    # pool by note
    by_note_probs, by_note_label = {}, {}
    for nid, pr, y in zip(note_ids, p1, labels):
        nid = int(nid)
        by_note_probs.setdefault(nid, []).append(float(pr))
        by_note_label[nid] = int(y)

    if POOL == "max":
        note_probs = np.array([max(v) for nid, v in sorted(by_note_probs.items())])
    else:
        note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
    note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

    # rank metrics
    try:
        auroc = roc_auc_score(note_labels, note_probs)
    except Exception:
        auroc = float("nan")
    try:
        auprc = average_precision_score(note_labels, note_probs)
    except Exception:
        auprc = float("nan")

    # find best F1 threshold on this eval set
    precs, recs, ths = precision_recall_curve(note_labels, note_probs)
    f1s = 2 * precs * recs / (precs + recs + 1e-12)
    best_ix = int(np.nanargmax(f1s))
    best_th = ths[best_ix-1] if best_ix > 0 and best_ix <= len(ths) else 0.5
    target_p = 0.80
    ix_p = np.where(precs >= target_p)[0]

    # report metrics at best_th
    note_pred_best = (note_probs >= best_th).astype(int)
    acc = accuracy_score(note_labels, note_pred_best)
    prec, rec, f1, _ = precision_recall_fscore_support(note_labels, note_pred_best, average="binary", zero_division=0)
    
    print({
      "true_pos_rate": float(np.mean(note_labels)),
      "pred_pos_rate@0.5": float(np.mean((note_probs >= 0.5).astype(int))),
      "mean_prob_pos": float(np.mean(note_probs[note_labels==1])) if np.any(note_labels==1) else None,
      "mean_prob_neg": float(np.mean(note_probs[note_labels==0])) if np.any(note_labels==0) else None,
      "best_th": float(best_th),
    })

    return {
        "note_auroc": auroc,
        "note_auprc": auprc,
        "note_f1": f1,
        "note_precision": prec,
        "note_recall": rec,
        "note_acc": acc,
        "th_best_f1": float(best_th),
    }

In [15]:
# --- Model Training ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from transformers import TrainerCallback
import time

trainer = NoteAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_note_level,
    )
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Note Auroc,Note Auprc,Note F1,Note Precision,Note Recall,Note Acc,Th Best F1
2000,0.542700,0.557723,0.716325,0.817724,0.832439,0.733793,0.961727,0.738386,0.569254
4000,0.531100,0.540274,0.742518,0.838199,0.836674,0.736655,0.968119,0.744601,0.347759
6000,0.532300,0.545146,0.752291,0.845076,0.839518,0.743655,0.963754,0.751027,0.580293
8000,0.561200,0.516540,0.761410,0.854406,0.841903,0.741830,0.973186,0.753029,0.402069
10000,0.531700,0.517763,0.766495,0.858976,0.842745,0.749772,0.962039,0.757400,0.331012
12000,0.522100,0.514930,0.769315,0.861384,0.842551,0.744940,0.969600,0.755135,0.425367
14000,0.517200,0.512752,0.772406,0.862097,0.843328,0.750791,0.961883,0.758506,0.305335
16000,0.489600,0.512665,0.777463,0.867390,0.844184,0.760806,0.948086,0.763510,0.448426
18000,0.526900,0.505208,0.777933,0.866837,0.845710,0.762734,0.948944,0.766038,0.457500
20000,0.509700,0.518518,0.772868,0.863526,0.844785,0.747421,0.971315,0.758822,0.331887


{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.9013483619509112, 'mean_prob_pos': 0.7633409203132601, 'mean_prob_neg': 0.6226970515606736, 'best_th': 0.5692541599273682}
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.8334035605182766, 'mean_prob_pos': 0.6854234924314153, 'mean_prob_neg': 0.4978719306863156, 'best_th': 0.34775859117507935}
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.897398082797851, 'mean_prob_pos': 0.8152119574842003, 'mean_prob_neg': 0.620928315637884, 'best_th': 0.5802926421165466}
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.8516275150110608, 'mean_prob_pos': 0.742392000147485, 'mean_prob_neg': 0.5266748108121023, 'best_th': 0.40206924080848694}
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.8165490361318867, 'mean_prob_pos': 0.7462881956650491, 'mean_prob_neg': 0.4883023849680151, 'best_th': 0.3310116231441498}
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.8593173917623512, 'm

TrainOutput(global_step=57255, training_loss=0.499538506546267, metrics={'train_runtime': 17335.1156, 'train_samples_per_second': 26.424, 'train_steps_per_second': 3.303, 'total_flos': 4.852891838378016e+16, 'train_loss': 0.499538506546267, 'epoch': 2.9998821109990437})

In [16]:
test_ds = StreamingNotes(
    "mv_test_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

In [17]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use test dataset
eval_loader = trainer.get_eval_dataloader(test_ds)

# number of test examples
n_test = len(test_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))


n_test=19082, bsz=2, total_steps=9541
Predicting: 100%|##########| 9541/9541 [01:40<00:00, 95.23it/s]
{'true_pos_rate': 0.6746550089539661, 'pred_pos_rate@0.5': 0.7694617086274097, 'mean_prob_pos': 0.7798824166046255, 'mean_prob_neg': 0.4700989341829198, 'best_th': 0.17866908013820648}

Test metrics: {'note_auroc': 0.7844556325130234, 'note_auprc': 0.8706412757516132, 'note_f1': 0.84687638760802, 'note_precision': 0.7527932960893855, 'note_recall': 0.9678351159341089, 'note_acc': 0.763878647424418, 'th_best_f1': 0.17866908013820648}

Confusion matrix:
[[ 2106  4071]
 [  412 12397]]

Classification report:
              precision    recall  f1-score   support

           0     0.8364    0.3409    0.4844      6177
           1     0.7528    0.9678    0.8469     12809

    accuracy                         0.7639     18986
   macro avg     0.7946    0.6544    0.6656     18986
weighted avg     0.7800    0.7639    0.7290     18986



In [18]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       1         0.709161
1          1       0         0.967349
2          2       1         0.662742
3          3       1         0.973620
4          4       0         0.677587


In [19]:
pred_df

,sample_id,y_true,pred_prob_notes
0,0,1,0.709161
1,1,0,0.967349
2,2,1,0.662742
3,3,1,0.973620
4,4,0,0.677587
...,...,...,...
18981,18981,1,0.799892
18982,18982,0,0.064653
18983,18983,0,0.683479
18984,18984,1,0.973418


In [20]:
from sklearn.metrics import roc_auc_score

auroc = roc_auc_score(note_labels, note_probs)
print("Note-level AUROC:", auroc)

Note-level AUROC: 0.7844556325130234


In [21]:
pred_df.to_csv("notes_test_predictions_task5.csv", index=False)

In [22]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use validation dataset
eval_loader = trainer.get_eval_dataloader(val_ds)

# number of validation examples
n_test = len(val_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

# concatenate predictions
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))

n_test=19091, bsz=2, total_steps=9546
Predicting: 100%|##########| 9546/9546 [01:39<00:00, 96.20it/s]
{'true_pos_rate': 0.6757084167281154, 'pred_pos_rate@0.5': 0.7742020436110818, 'mean_prob_pos': 0.7822969124552772, 'mean_prob_neg': 0.4754377066373912, 'best_th': 0.2287159264087677}

Test metrics: {'note_auroc': 0.7825057765308173, 'note_auprc': 0.870506207321025, 'note_f1': 0.8458019468235132, 'note_precision': 0.7568948534843635, 'note_recall': 0.9583755553823369, 'note_acc': 0.763878647424418, 'th_best_f1': 0.2287159264087677}

Confusion matrix:
[[ 2208  3949]
 [  534 12295]]

Classification report:
              precision    recall  f1-score   support

           0     0.8053    0.3586    0.4962      6157
           1     0.7569    0.9584    0.8458     12829

    accuracy                         0.7639     18986
   macro avg     0.7811    0.6585    0.6710     18986
weighted avg     0.7726    0.7639    0.7324     18986



In [23]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       1         0.977522
1          1       1         0.838746
2          2       1         0.680587
3          3       1         0.815637
4          4       1         0.895979


In [24]:
pred_df.to_csv("notes_val_predictions_task5.csv", index=False)